# Chapter 11: CI for Python: Lint, Test, Coverage Matrix (notebook edition)

## Learning Objectives

- Compute coverage and gate it
- Explain cache saves and hits
- Detect the defaults trap

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Coverage arithmetic

The first real run covered 9 of 17 statements. Compute it and compare with the gate.

In [ ]:
import json
s = json.loads((ROOT/"fixtures"/"ci_ch11_summary.json").read_text())
def coverage_percent(statements, missed):
    return 100 * (statements - missed) / statements
red = s["runs"]["red_gate"]["coverage"]
print(round(coverage_percent(red["statements"], red["missed"]), 2), "vs gate", red["gate"])
assert round(coverage_percent(17, 8), 2) == 52.94 and coverage_percent(17, 8) < 80

## 2. The gate rule

`ci-ok` passes only when every upstream result is exactly `success`; `skipped` and `cancelled` must block too.

In [ ]:
def gate(lint, test):
    return lint == "success" and test == "success"
for combo in [("success","success"),("failure","success"),("skipped","success"),("success","cancelled")]:
    print(combo, gate(*combo))
assert gate("success","success") and not gate("skipped","success")
print(s["runs"]["bad_lint"]["ci_ok_report"])

## 3. When was the pip cache saved?

A failing job saves nothing; later runs hit.


In [ ]:
print(s["runs"]["red_gate"]["pip_cache_saved_by"])
for rid, outcome in s["cache_outcomes"].items():
    print(rid, outcome)
assert s["runs"]["red_gate"]["pip_cache_saved_by"] == ["lint"]
assert set(s["cache_outcomes"]["37321351215"].values()) == {"hit"}

## 4. Did the cache help?

Cold vs warm `pip install` seconds per Python version.

In [ ]:
cold, warm = s["install_seconds"]["cold"], s["install_seconds"]["warm"]
print({k: (cold[k], warm[k]) for k in cold})
assert max(cold[k] - warm[k] for k in cold) <= 1

## 5. Detect the working-directory trap statically

The workflow-level default hit a job with no checkout.

In [ ]:
from intro_gha import WORKFLOWS_DIR
from intro_gha.workflow import jobs_missing_checkout_with_workflow_workdir, load_workflow
buggy = {"defaults": {"run": {"working-directory": "sandbox/tally"}}, "jobs": {"ci-ok": {"steps": [{"run": "true"}]}}}
print(jobs_missing_checkout_with_workflow_workdir(buggy))
assert jobs_missing_checkout_with_workflow_workdir(buggy) == ["ci-ok"]
assert jobs_missing_checkout_with_workflow_workdir(load_workflow(WORKFLOWS_DIR / "ch11-python-ci.yml")) == []

## Takeaways & Next Steps

- Gate on exact `success`; use `always()`.
- Workflow-level `working-directory` hits every job.
- Measure before caching.
- Next: Chapter 12, containers and GHCR.

## Chapter Link

Read: `learning_modules/chapter_11_ci_for_python.md`